In [12]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [13]:
import warnings
warnings.filterwarnings('ignore')

In [14]:
train_df = pd.read_csv('../data/processed/train.csv')
val_df = pd.read_csv('../data/processed/val.csv')
test_df = pd.read_csv('../data/processed/test.csv')

In [15]:
targets = ['Sleep_Quality_Num', 'Stress_Level_Num', 'Health_Issues_Num']

In [16]:
X_train = train_df.drop(columns=targets)
y_train = train_df[targets]

X_val = val_df.drop(columns=targets)
y_val = val_df[targets]

X_test = test_df.drop(columns=targets)
y_test = test_df[targets]

print(f"Data Loaded! X_train shape: {X_train.shape}")
print(f"Targets ready: {targets}")

Data Loaded! X_train shape: (7000, 44)
Targets ready: ['Sleep_Quality_Num', 'Stress_Level_Num', 'Health_Issues_Num']


In [17]:
baseline_results = []

def train_and_evaluate(model, model_name, X_train, y_train, X_val, y_val, target_name):
    print(f"\n{'='*50}")
    print(f"Training {model_name} for {target_name}")
    print(f"{'='*50}")
    
    # 1. Train the model
    model.fit(X_train, y_train)
    
    # 2. Make predictions on the VALIDATION set
    y_pred = model.predict(X_val)
    
    # 3. Calculate metrics
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred, average='macro')
    
    print(f"Validation Accuracy: {acc:.4f}")
    print(f"Validation F1 Score (Macro): {f1:.4f}\n")
    print(classification_report(y_val, y_pred))
    
    # 4. Save the model
    os.makedirs('../models/baselines', exist_ok=True)
    model_path = f'../models/baselines/{model_name.lower().replace(" ", "_")}_{target_name}.joblib'
    joblib.dump(model, model_path)
    print(f"Model saved to: {model_path}")
    
    # 5. Log results for final summary
    baseline_results.append({
        'Target': target_name,
        'Model': model_name,
        'Accuracy': acc,
        'F1_Macro': f1
    })
    
    return model

In [18]:
for target in targets:
    lr_model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
    
    train_and_evaluate(
        model=lr_model,
        model_name="Logistic Regression",
        X_train=X_train,
        y_train=y_train[target],
        X_val=X_val,
        y_val=y_val[target],
        target_name=target
    )


Training Logistic Regression for Sleep_Quality_Num
Validation Accuracy: 0.9640
Validation F1 Score (Macro): 0.9600

              precision    recall  f1-score   support

         0.0       0.95      1.00      0.97       144
         1.0       0.94      0.97      0.96       307
         2.0       1.00      0.95      0.97       846
         3.0       0.89      1.00      0.94       203

    accuracy                           0.96      1500
   macro avg       0.94      0.98      0.96      1500
weighted avg       0.97      0.96      0.96      1500

Model saved to: ../models/baselines/logistic_regression_Sleep_Quality_Num.joblib

Training Logistic Regression for Stress_Level_Num
Validation Accuracy: 0.9833
Validation F1 Score (Macro): 0.9759

              precision    recall  f1-score   support

         0.0       1.00      0.98      0.99      1049
         1.0       0.94      0.98      0.96       307
         2.0       0.95      1.00      0.98       144

    accuracy                     

In [ ]:
for target in targets:
    lgbm_model = LGBMClassifier(random_state=42, class_weight='balanced', verbose=-1)
    
    train_and_evaluate(
        model=lgbm_model,
        model_name="LightGBM",
        X_train=X_train,
        y_train=y_train[target],
        X_val=X_val,
        y_val=y_val[target],
        target_name=target
    )


Training LightGBM for Sleep_Quality_Num
Validation Accuracy: 0.9693
Validation F1 Score (Macro): 0.9655

              precision    recall  f1-score   support

         0.0       0.96      0.99      0.98       144
         1.0       0.96      0.95      0.96       307
         2.0       0.98      0.98      0.98       846
         3.0       0.95      0.96      0.95       203

    accuracy                           0.97      1500
   macro avg       0.96      0.97      0.97      1500
weighted avg       0.97      0.97      0.97      1500

Model saved to: ../models/baselines/lightgbm_Sleep_Quality_Num.joblib

Training LightGBM for Stress_Level_Num
Validation Accuracy: 0.9833
Validation F1 Score (Macro): 0.9766

              precision    recall  f1-score   support

         0.0       0.99      0.99      0.99      1049
         1.0       0.95      0.96      0.96       307
         2.0       0.97      0.99      0.98       144

    accuracy                           0.98      1500
   macro avg

In [ ]:
results_df = pd.DataFrame(baseline_results)

results_df = results_df.sort_values(by=['Target', 'F1_Macro'], ascending=[True, False]).reset_index(drop=True)

print("--- FINAL BASELINE METRICS ---")
display(results_df)

os.makedirs('../metrics', exist_ok=True)
results_df.to_csv('../metrics/baseline_metrics.csv', index=False)
print("\n✓ Baseline metrics successfully saved to '../metrics/baseline_metrics.csv'")

--- FINAL BASELINE METRICS ---


,Target,Model,Accuracy,F1_Macro
0,Health_Issues_Num,LightGBM,0.984667,0.988111
1,Health_Issues_Num,Logistic Regression,0.801333,0.697665
2,Sleep_Quality_Num,LightGBM,0.969333,0.965507
3,Sleep_Quality_Num,Logistic Regression,0.964000,0.960028
4,Stress_Level_Num,LightGBM,0.983333,0.976616
5,Stress_Level_Num,Logistic Regression,0.983333,0.975872



✓ Baseline metrics successfully saved to '../metrics/baseline_metrics.csv'
